# Packages

In [1]:
import random
import numpy as np
import pandas as pd
import heapq
from collections import deque
from collections import Counter
import copy
import time

# Rubik Model

### directions:
![Alt text for the image](directions.png)

### rubik initial color direction:
![Alt text for the image](cube.png)

n -> negative. we have 6 color for 6 face. colors -> white, yellow, blue, green, red, orange. also have black color for inside faces

In [2]:
class Cube:
    def __init__(self, x: str, nx: str, y: str, ny: str, z: str, nz: str):
        self.x = x
        self.nx = nx
        self.y = y
        self.ny = ny
        self.z = z
        self.nz = nz


class Rubik:
    def __init__(self, x: int, y: int, z: int):
        self.x = x
        self.y = y
        self.z = z
        self.cubes = np.empty((x, y, z), dtype=Cube)
        self.initialize_rubik()

    def initialize_rubik(self):
        for i in range(self.x):
            for j in range(self.y):
                for k in range(self.z):
                    x_color = 'blue' if i == self.x - 1 else 'black'
                    nx_color = 'green' if i == 0 else 'black'
                    y_color = 'white' if j == self.y - 1 else 'black'
                    ny_color = 'yellow' if j == 0 else 'black'
                    z_color = 'red' if k == self.z - 1 else 'black'
                    nz_color = 'orange' if k == 0 else 'black'
                    self.cubes[i, j, k] = Cube(x_color, nx_color, y_color, ny_color, z_color, nz_color)

    def __eq__(self, other):
        x_size, y_size, z_size = self.x, self.y, self.z
        if not all(self.cubes[x_size - 1, j, k].x == other.cubes[x_size - 1, j, k].x
                   for j in range(y_size) for k in range(z_size)):
            return False
        if not all(self.cubes[0, j, k].nx == other.cubes[0, j, k].nx
                   for j in range(y_size) for k in range(z_size)):
            return False
        if not all(self.cubes[i, y_size - 1, k].y == other.cubes[i, y_size - 1, k].y
                   for i in range(x_size) for k in range(z_size)):
            return False
        if not all(self.cubes[i, 0, k].ny == other.cubes[i, 0, k].ny
                   for i in range(x_size) for k in range(z_size)):
            return False
        if not all(self.cubes[i, j, z_size - 1].z == other.cubes[i, j, z_size - 1].z
                   for i in range(x_size) for j in range(y_size)):
            return False
        if not all(self.cubes[i, j, 0].nz == other.cubes[i, j, 0].nz
                   for i in range(x_size) for j in range(y_size)):
            return False
        return True

    def __hash__(self):
        hash_list = []
        x_size, y_size, z_size = self.x, self.y, self.z
        for j in range(y_size):
            for k in range(z_size):
                hash_list.append(self.cubes[x_size - 1, j, k].x)
        for j in range(y_size):
            for k in range(z_size):
                hash_list.append(self.cubes[0, j, k].nx)
        for i in range(x_size):
            for k in range(z_size):
                hash_list.append(self.cubes[i, y_size - 1, k].y)
        for i in range(x_size):
            for k in range(z_size):
                hash_list.append(self.cubes[i, 0, k].ny)
        for i in range(x_size):
            for j in range(y_size):
                hash_list.append(self.cubes[i, j, z_size - 1].z)
        for i in range(x_size):
            for j in range(y_size):
                hash_list.append(self.cubes[i, j, 0].nz)
        return hash(tuple(hash_list))

# Functions

### row reverse function:

In [3]:
def row_reverse(rubik: Rubik, face: str, a: int, b: int):
    rubik_copy = copy.deepcopy(rubik)
    if face == 'x':
        l = int(rubik_copy.x)
        for c in range(l // 2):
            temp = rubik_copy.cubes[c, a, b]
            rubik_copy.cubes[c, a, b] = rubik_copy.cubes[l - c - 1, a, b]
            rubik_copy.cubes[l - c - 1, a, b] = temp
        for c in range(l):
            cube = rubik_copy.cubes[c, a, b]
            cube.x, cube.nx = cube.nx, cube.x
    elif face == 'y':
        l = int(rubik_copy.y)
        for c in range(l // 2):
            temp = rubik_copy.cubes[a, c, b]
            rubik_copy.cubes[a, c, b] = rubik_copy.cubes[a, l - c - 1, b]
            rubik_copy.cubes[a, l - c - 1, b] = temp
        for c in range(l):
            cube = rubik_copy.cubes[a, c, b]
            cube.y, cube.ny = cube.ny, cube.y
    elif face == 'z':
        l = int(rubik_copy.z)
        for c in range(l // 2):
            temp = rubik_copy.cubes[a, b, c]
            rubik_copy.cubes[a, b, c] = rubik_copy.cubes[a, b, l - c - 1]
            rubik_copy.cubes[a, b, l - c - 1] = temp
        for c in range(l):
            cube = rubik_copy.cubes[a, b, c]
            cube.z, cube.nz = cube.nz, cube.z
    return rubik_copy

### print rubik:

In [4]:
def print_rubik(rubik: Rubik):
    map_color = {
        "black": "30",
        "red": "31",
        "green": "32",
        "yellow": "33",
        "blue": "34",
        "orange": "38;5;208",
        "white": "37",
    }

    def c(col):
        return f'\033[{map_color[col]}mo\033[0m'

    x_size, y_size, z_size = rubik.x, rubik.y, rubik.z
    #Top face
    for i in range(x_size):
        print(' ', end='')
        print(" " * (z_size * 2), end='')  # padding to center
        for j in range(z_size):
            print(c(rubik.cubes[i, y_size - 1, j].y), end=' ')
        print()
    #Left, Front, Right, Back faces
    for j in range(y_size):
        # Left face
        for k in range(z_size):
            print(c(rubik.cubes[0, j, k].nx), end=' ')
        print(" ", end='')
        # Front face
        for i in range(x_size):
            print(c(rubik.cubes[i, j, z_size - 1].z), end=' ')
        print(" ", end='')
        # Right face
        for k in range(z_size):
            print(c(rubik.cubes[x_size - 1, j, k].x), end=' ')
        print(" ", end='')
        # Back face
        for i in range(x_size):
            print(c(rubik.cubes[i, j, 0].nz), end=' ')
        print()
    #Bottom face
    for i in range(x_size):
        print(' ', end='')
        print(" " * (z_size * 2), end='')  # padding to center
        for j in range(z_size):
            print(c(rubik.cubes[i, 0, j].ny), end=' ')
        print()

### shuffle rubik randomly:

give number t and call row_reverse() functon N time with random attribute.

In [5]:
def shuffle_rubik(rubik: Rubik, t: int):
    seed = 1384
    random.seed(seed)
    faces = ['x', 'y', 'z']
    for _ in range(t):
        face = random.choice(faces)
        a, b = (0, 0)
        if face == 'x':
            a = random.randint(0, rubik.y - 1)
            b = random.randint(0, rubik.z - 1)
        elif face == 'y':
            a = random.randint(0, rubik.x - 1)
            b = random.randint(0, rubik.z - 1)
        elif face == 'z':
            a = random.randint(0, rubik.x - 1)
            b = random.randint(0, rubik.y - 1)
        rubik = row_reverse(rubik, face, a, b)
    return rubik

# State Space

### node model:

In [6]:
class Node:
    def __init__(self, rubik: Rubik, parent=None, depth=0, path_cost=0):
        self.rubik = rubik
        self.parent = parent
        self.depth = depth
        self.path_cost = path_cost
        self.f = 0

    def __eq__(self, other):
        return self.rubik == other.rubik

    def __lt__(self, other):
        return False

    def __hash__(self):
        return hash(self.rubik)

### expand node:

try-except is for handle that the node don't expand its father for first time

In [7]:
def expand(node: Node):
    children = []
    moves = [
        ('x', node.rubik.y, node.rubik.z),
        ('y', node.rubik.x, node.rubik.z),
        ('z', node.rubik.x, node.rubik.y),
    ]
    for face, a, b in moves:
        for i in range(a):
            for j in range(b):
                child_rubik = row_reverse(node.rubik, face, i, j)
                child = Node(child_rubik, node, node.depth + 1, node.path_cost + 1)
                if node.parent is None or child != node.parent:
                    children.append(child)
    return children

### problem model:

In [8]:
class Problem:
    def __init__(self, initial: Node, goals: list[Node]):
        self.initial = initial
        self.goals = set(goals)

### make problem goals:

In [9]:
def get_goals(x: int, y: int, z: int):
    r1 = Rubik(x, y, z)
    goals = [Node(r1)]

    r2 = copy.deepcopy(r1)
    for i in range(y):
        for j in range(z):
            r2 = row_reverse(r2, 'x', i, j)
    goals.append(Node(r2))
    r3 = copy.deepcopy(r2)
    for i in range(x):
        for j in range(z):
            r3 = row_reverse(r3, 'y', i, j)
    goals.append(Node(r3))
    r4 = copy.deepcopy(r3)
    for i in range(y):
        for j in range(z):
            r4 = row_reverse(r4, 'x', i, j)
    goals.append(Node(r4))
    r5 = copy.deepcopy(r4)
    for i in range(x):
        for j in range(y):
            r5 = row_reverse(r5, 'z', i, j)
    goals.append(Node(r5))
    r6 = copy.deepcopy(r5)
    for i in range(y):
        for j in range(z):
            r6 = row_reverse(r6, 'x', i, j)
    goals.append(Node(r6))
    r7 = copy.deepcopy(r6)
    for i in range(x):
        for j in range(z):
            r7 = row_reverse(r7, 'y', i, j)
    goals.append(Node(r7))
    r8 = copy.deepcopy(r7)
    for i in range(y):
        for j in range(z):
            r8 = row_reverse(r8, 'x', i, j)
    goals.append(Node(r8))

    return goals

# Best-First Search

all_expanded_nodes need for count number of expanded nodes.

In [10]:
all_expanded_nodes = 0


def Best_First_Search(problem: Problem, f):
    frontier = []
    reached = {problem.initial.rubik: problem.initial}
    heapq.heappush(frontier, (0, problem.initial))
    while frontier:
        node = heapq.heappop(frontier)[1]
        if node in problem.goals:
            return node
        for child in expand(node):
            s = child.rubik
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                heapq.heappush(frontier, (f(child), child))
                global all_expanded_nodes
                all_expanded_nodes += 1
    return None

# Uninformed Algorithms

we implement best-first search. now for BFS, DFS, UCS just need to  define f function for each algorithm.
in this problem the UCS and BFS are the same f function.

## Breadth-First Search (BFS):

In [11]:
def f_bfs(node: Node):
    return node.depth


def Breadth_First_Search(problem: Problem):
    frontier = deque()
    reached = {problem.initial.rubik: problem.initial}
    frontier.append(problem.initial)
    while frontier:
        node = frontier.popleft()

        for child in expand(node):
            s = child.rubik
            if child in problem.goals:
                return child
            if s not in reached:
                reached[s] = child
                frontier.append(child)
                global all_expanded_nodes
                all_expanded_nodes += 1
    return None

## Depth-First Search (DFS):

In [12]:
def f_dfs(node: Node):
    return -1 * node.depth


def Depth_First_Search(problem: Problem):
    frontier = []
    reached = {problem.initial.rubik: problem.initial}
    frontier.append(problem.initial)
    while frontier:
        node = frontier.pop()
        for child in expand(node):
            s = child.rubik
            if child in problem.goals:
                return child
            if s not in reached:
                reached[s] = child
                frontier.append(child)
                global all_expanded_nodes
                all_expanded_nodes += 1
    return None

## Uniform Cost Search (UCS):

In [13]:
def f_ucs(node: Node):
    return node.path_cost


def Uniform_Cost_Search(problem: Problem):
    return Best_First_Search(problem, f_ucs)

## Depth-Limited Search (DLS):

In [14]:
def Depth_Limited_Search(problem: Problem, l):
    frontier = []
    reached = {problem.initial.rubik: problem.initial}
    frontier.append(problem.initial)
    res = None
    while frontier:
        node = frontier.pop()
        if node.depth > l:
            res = 'cutoff'
        else:
            for child in expand(node):
                s = child.rubik
                if child in problem.goals:
                    return child
                if s not in reached:
                    reached[s] = child
                    frontier.append(child)
                    global all_expanded_nodes
                    all_expanded_nodes += 1

    return res

## Iterative Deepening Search (IDS):

In [15]:
def Iterative_Deepening_Search(problem: Problem):
    depth = 0
    while True:
        res = Depth_Limited_Search(problem, depth)
        depth += 1
        if type(res) != str:
            return res


# Informed Algorithms

we implement best-first search. now for A*, weighted A*, Greedy best just need to  define f function for each algorithm.
first we implement heuristic function h for informed approach.

## Heuristic Function:

in this heuristic, function first we count colors of each 6 faces to find majority color. then count the number of cube in each face that has different color from that face majority color and assign it to "dif". finally we divide "dif" by 2, because each move at least make 2 color in their place (in a move just two color go to another faces and other colors stay at previous face). this division guarantees that h is consistent and as a result it is admissible.<br>
note that with this approach of rubik moves, in each face at most we have two colors; because to move a color to another face we just can reverse row, which moves color to its mirror face. it means in each face we just have white/yellow or red/orange or blue/green color.

In [16]:
def h(node: Node):
    rubik = node.rubik
    dif = 0
    faces = [
        [rubik.cubes[rubik.x - 1, j, k].x for j in range(rubik.y) for k in range(rubik.z)],
        [rubik.cubes[0, j, k].nx for j in range(rubik.y) for k in range(rubik.z)],
        [rubik.cubes[i, rubik.y - 1, k].y for i in range(rubik.x) for k in range(rubik.z)],
        [rubik.cubes[i, 0, k].ny for i in range(rubik.x) for k in range(rubik.z)],
        [rubik.cubes[i, j, rubik.z - 1].z for i in range(rubik.x) for j in range(rubik.y)],
        [rubik.cubes[i, j, 0].nz for i in range(rubik.x) for j in range(rubik.y)],
    ]
    for face in faces:
        if not face:
            continue
        majority = Counter(face).most_common(1)[0][0]
        dif += len(face) - face.count(majority)
    return dif // 2

## Greedy Best-First Search (GBFS):

In [17]:
def f_gbfs(node: Node):
    return h(node)


def Greedy_Best_First_Search(problem: Problem):
    return Best_First_Search(problem, f_gbfs)

## A* Search:

In [18]:
def f_astar(node: Node):
    return h(node) + node.path_cost


def A_star(problem: Problem):
    return Best_First_Search(problem, f_astar)

## Weighted A* Search:

In [19]:
def f_weighted_astar(weight: int):
    def f_w(node: Node):
        return weight * h(node) + node.path_cost

    return f_w


def Weighted_A_Star(problem: Problem, weight: int):
    return Best_First_Search(problem, f_weighted_astar(weight))

## Iterative Deepening A* (IDA*):

In [20]:
def Iterative_Deepening_A_star(problem):
    def find(node, g, bound, path_set):
        f = g + h(node)
        if f > bound:
            return f
        if node in problem.goals:
            return node
        path_set.add(node)
        min_next = float('inf')
        for child in expand(node):
            if child not in path_set:
                res = find(child, g + 1, bound, path_set)
                if type(res) is Node:
                    return res
                min_next = min(min_next, res)
            global all_expanded_nodes
            all_expanded_nodes += 1
        path_set.remove(node)
        return min_next

    bound = h(problem.initial)
    while True:
        path_set = set()
        res = find(problem.initial, 0, bound, path_set)
        if isinstance(res, Node):
            return res
        if res == float('inf'):
            return None
        bound = res

## Recursive Best-First Search (RBFS):

In [21]:
def Recursive_Best_First_Search(problem: Problem):
    def RBFS(problem2: Problem, node: Node, f_limit: float):
        if node in problem2.goals:
            return node, node.f
        children = expand(node)
        if not children:
            return None, float('inf')
        for child in children:
            child_h = h(child)
            child.f = max(child.path_cost + child_h, node.f)
            global all_expanded_nodes
            all_expanded_nodes += 1
        while True:
            children.sort(key=lambda ch: ch.f)
            best1 = children[0]
            if best1.f > f_limit:
                return None, best1.f
            best2 = children[1].f if len(children) > 1 else float('inf')
            res, best1.f = RBFS(problem2, best1, min(f_limit, best2))
            if res is not None:
                return res, best1.f

    problem.initial.f = h(problem.initial)
    solution = RBFS(problem, problem.initial, float('inf'))[0]
    return solution

# Analysis 10 Algorithms

### hints for analysis:

- if you want to get path of solution, call print_rubik(node) for all parents of result_i<br>
print_rubik(result_i)<br>
prunt_rubik(result_i.parent)<br>
prunt_rubik(result_i.parent.parent)<br>
...<br>
...<br>
...<br>

- if you want to change l for Depth_Limited or change wight for Weighted_A_star, change attribute of these functions in below block.

- this block is for compare all algorithms. if you want to analysis just some algorithms for the same input, comment other result_i in this block:

In [23]:
data = {
    "Algorithms": ["BFS", "DFS", "UCS", "DLS", "IDS", "GBFS", "A*", "W-A*", "IDA*", "RBFS"],
    "Time(s)": ['', '', '', '', '', '', '', '', '', ''],
    "Expanded Nodes": ['', '', '', '', '', '', '', '', '', ''],
    "Depth of Solution": ['', '', '', '', '', '', '', '', '', '']
}
df = pd.DataFrame(data)

x = int(input("print your x size:"))
y = int(input("print your y size:"))
z = int(input("print your z size:"))
s = int(input("number of shuffle for shuffling rubik:"))
rubik = Rubik(x, y, z)
shuffled_rubik = shuffle_rubik(rubik, s)
root_node = Node(shuffled_rubik)
problem = Problem(root_node, get_goals(x, y, z))

#BFS
t = time.time()
all_expanded_nodes = 0
result0 = Breadth_First_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[0] = ["BFS", t, a, result0.depth]
print(df.loc[0])
print()

#DFS
t = time.time()
all_expanded_nodes = 0
result1 = Depth_First_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[1] = ["DFS", t, a, result1.depth]
print(df.loc[1])
print()

#UCS
t = time.time()
all_expanded_nodes = 0
result2 = Uniform_Cost_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[2] = ["UCS", t, a, result2.depth]
print(df.loc[2])
print()

#DLS
t = time.time()
all_expanded_nodes = 0
result3 = Depth_Limited_Search(problem, 5)
a = all_expanded_nodes
t = time.time() - t
df.loc[3] = ["DLS", t, a, result3.depth]
print(df.loc[3])
print()

#IDS
t = time.time()
all_expanded_nodes = 0
result4 = Iterative_Deepening_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[4] = ["IDS", t, a, result4.depth]
print(df.loc[4])
print()

#GBFS
t = time.time()
all_expanded_nodes = 0
result5 = Greedy_Best_First_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[5] = ["GBFS", t, a, result5.depth]
print(df.loc[5])
print()

#A*
t = time.time()
all_expanded_nodes = 0
result6 = A_star(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[6] = ["A*", t, a, result6.depth]
print(df.loc[6])
print()

#W-A*
t = time.time()
all_expanded_nodes = 0
result7 = Weighted_A_Star(problem, 5)
a = all_expanded_nodes
t = time.time() - t
df.loc[7] = ["W-A*", t, a, result7.depth]
print(df.loc[7])
print()

#IDA*
t = time.time()
all_expanded_nodes = 0
result8 = Iterative_Deepening_A_star(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[8] = ["IDA*", t, a, result8.depth]
print(df.loc[8])
print()

#RBFS
t = time.time()
all_expanded_nodes = 0
result9 = Recursive_Best_First_Search(problem)
a = all_expanded_nodes
t = time.time() - t
df.loc[9] = ["RBFS", t, a, result9.depth]
print(df.loc[9])
print()

display(df)


Algorithms                BFS
Time(s)              0.985056
Expanded Nodes           1237
Depth of Solution           3
Name: 0, dtype: object

Algorithms                 UCS
Time(s)              13.038905
Expanded Nodes           18128
Depth of Solution            3
Name: 2, dtype: object

Algorithms                IDS
Time(s)              4.560728
Expanded Nodes           4323
Depth of Solution           3
Name: 4, dtype: object

Algorithms               GBFS
Time(s)              0.027983
Expanded Nodes             78
Depth of Solution           3
Name: 5, dtype: object

Algorithms                 A*
Time(s)              0.044469
Expanded Nodes            128
Depth of Solution           3
Name: 6, dtype: object

Algorithms               W-A*
Time(s)              0.025656
Expanded Nodes             78
Depth of Solution           3
Name: 7, dtype: object

Algorithms               IDA*
Time(s)              0.023789
Expanded Nodes             20
Depth of Solution           3
Name: 8, dty

,Algorithms,Time(s),Expanded Nodes,Depth of Solution
0,BFS,0.985056,1237,3
1,DFS,,,
2,UCS,13.038905,18128,3
3,DLS,,,
4,IDS,4.560728,4323,3
5,GBFS,0.027983,78,3
6,A*,0.044469,128,3
7,W-A*,0.025656,78,3
8,IDA*,0.023789,20,3
9,RBFS,0.023369,79,3


# Conclusion

### Uninformed Algorithms:

these type of algorithms work for this problem very hard. these just work for small x, y, z and small shuffle times.
for example <B>BFS</B> work for 3x3x3 at most shuffle 4 times (because take very large memory and make the computation very slow). <B>DFS</B> is so worse than <B>BFS</B> and just works on 2x2x2 (because the number of state in 3x3x3 rubik is very large (2^27) and it is impossible to see all nodes unless we are lucky!). by experience <B>IDS</B> often work more efficient between these algorithms.

### Informed Algorithms

with good heuristic function these algorithms are so more efficient than uninformed algorithms. <B>GBFS</B> is fast but
 maybe gives a very non-optimal path and doesn't suggest. <B>A*</B> gives optimal path but for large input uses lots of memory. <B>Weighted A*</B> gives path with error in range of weight, we can control the error and it is acceptable. <B>RBFS</B> works near to <B>Weighted A*</B>, more memory but optimal path. at the end we have <B>IDA*</B> that works well on each input size and also give the optimal path.